# Python for Poets 2026 - Workbook 3 - Finding distinctive words (v2)

Which words are typical of Democratic presidents' State of the Union speeches, and which of Republican ones? In this workbook we'll answer that question step by step: counting words, comparing their frequencies, testing whether a difference is real, and then going back to read the words in context.

## Setup

In [ ]:
#@title Install and import the `python4poets` package
%pip install -q IProgress ipywidgets jupyter
try:
    import python4poets as p4p
except ImportError:
    # Not installed yet: install it into this notebook's Python, then import
    %pip install -q git+https://github.com/quadrismegistus/python-for-poets
    import python4poets as p4p

import pandas as pd

Everything in the package is a function starting with `p4p.`

∇ *Make a new cell, type `p4p.` and wait (or press Tab). What functions are there? Then try `p4p.kwic?` to read the help for one of them, and `p4p.kwic??` to read its code.*

## Loading the corpus

In [ ]:
#@title Download and open the State of the Union corpus
df = p4p.load_sotu()
df

Each row is one speech. The `txt` column holds the speech's full text; the `party` column comes from a separate spreadsheet of presidents and their parties.

In [ ]:
#@title Show number of speeches per party
df['party'].value_counts()

In [ ]:
#@title TODO: Show number of speeches per president
df['REPLACE_ME'].value_counts()

## Counting words

To a computer, a text is just a long string of characters. Before we can count words, we have to split it into words: this is called *tokenizing*.

In [ ]:
#@title Split a sentence into words
p4p.tokenize("We the People of the United States, in Order to form a more perfect Union...")

∇ *What happened to the capital letters? To the punctuation? Try tokenizing a sentence with "don't" in it: what happens?*

The most common words in English ("the", "of", "and") are called *stopwords*. We can leave them out:

In [ ]:
#@title Tokenize, leaving out stopwords
p4p.tokenize("We the People of the United States, in Order to form a more perfect Union...", stopwords='english')

∇ *Which words disappeared? Is there any word among them that might matter in a political speech?*

In [ ]:
#@title Count the words in one speech
lincoln_1863 = df.query('id == "Lincoln_1863"').iloc[0]['txt']
lincoln_counts = p4p.count_words(lincoln_1863, stopwords='english')
lincoln_counts.most_common(20)

In [ ]:
#@title TODO: Count the words in another speech
speech = df.query('id == "REPLACE_ME"').iloc[0]['txt']
speech_counts = p4p.count_words(speech, stopwords='english')
speech_counts.most_common(20)

Now let's add up the counts for *all* the Democratic speeches, and for *all* the Republican ones. This time we'll keep the stopwords in: whether a party says "we" more than the other is itself interesting.

In [ ]:
#@title Count words for each party
counts = p4p.count_words_by_group(df, 'party')

counts_D = counts['Democrat']
counts_R = counts['Republican']

print('10 most common words for Democrats:')
print(counts_D.most_common(10))
print()
print('10 most common words for Republicans:')
print(counts_R.most_common(10))

## Relative frequency

Democrats and Republicans didn't speak the same total number of words, so raw counts aren't comparable. Instead we compare *relative* frequencies: how many times per million words (`fpm`) each word is used.

In [ ]:
#@title Compute relative frequencies

# get total counts
total_D = sum(counts_D.values())
total_R = sum(counts_R.values())
print(f'Democrats: {total_D:,} words. Republicans: {total_R:,} words.')

# get relative count (frequency per million or `fpm`)
fpm_D = {word: count / total_D * 1_000_000 for word, count in counts_D.items()}
fpm_R = {word: count / total_R * 1_000_000 for word, count in counts_R.items()}

# look up a single word
word = 'government'
fpm_D.get(word, 0), fpm_R.get(word, 0)

In [ ]:
#@title TODO: Try a different word
word = 'REPLACE_ME'
fpm_D.get(word, 0), fpm_R.get(word, 0)

## How different is "different"?

One way to measure how differently two parties use a word is to *subtract* one frequency from the other. Another is to *divide* one by the other. Let's compare two words.

In [ ]:
#@title Difference vs. ratio
for word in ['the', 'tariff']:
    difference = fpm_D[word] - fpm_R[word]
    ratio = fpm_D[word] / fpm_R[word]
    print(f'{word}:  Democrats {fpm_D[word]:,.1f}  Republicans {fpm_R[word]:,.1f}  difference {difference:,.1f}  ratio {ratio:.2f}')

∇ *By the difference, which word is more distinctive? By the ratio? Which measure matches your sense of which word is more "partisan"?*

Subtraction favours common words: a tiny proportional difference in "the" is still thousands of words per million. The ratio asks a better question: *how many times* more often does one party use the word?

Ratios have an awkward asymmetry, though: "twice as often" is 2, but "half as often" is 0.5. Taking the logarithm (base 2) makes them symmetric: twice as often is **+1**, half as often is **−1**, the same is **0**. This is the **log ratio**.

In [ ]:
#@title Compute the log ratio for a word
import math

word = 'immigration'
math.log2(fpm_D[word] / fpm_R[word])

∇ *Is "immigration" more Democratic or more Republican? By how much (how many times more often)?*

∇ *What would happen to the log ratio of a word that one party never uses at all?*

## Is the difference real?

If one party says a word 3 times and the other once, the ratio is large, but it could easily be chance. As in Workbook 2, we can test this with a contingency table: for each party, how many of its words are this word, and how many are not?

In [ ]:
#@title Contingency table for a word
def contingency_table_for_word(word):
    return pd.DataFrame({
        'Democrat': {
            'this word': counts_D.get(word, 0),
            'other words': total_D - counts_D.get(word, 0),
        },
        'Republican': {
            'this word': counts_R.get(word, 0),
            'other words': total_R - counts_R.get(word, 0),
        },
    })

contingency_table_for_word('immigration')

In Workbook 2 we tested tables like this with the chi-squared statistic. For word counts, linguists usually prefer a close cousin, the **log-likelihood** statistic, or **G²**, which is more reliable for words that are rare compared to the size of the corpus. Python computes it with the same function as before, with one extra option:

In [ ]:
#@title Statistical significance (G²) and observed/expected for a word
from scipy.stats import chi2_contingency

table = contingency_table_for_word('immigration')
g2, p, dof, expected = chi2_contingency(table, lambda_='log-likelihood', correction=False)

print(f'G²: {g2:.1f}')
print(f'P-value: {p}')

# observed / expected
table / expected

In [ ]:
#@title TODO: Test another word
table = contingency_table_for_word('REPLACE_ME')
g2, p, dof, expected = chi2_contingency(table, lambda_='log-likelihood', correction=False)

print(f'G²: {g2:.1f}')
print(f'P-value: {p}')
table / expected

A p-value below 0.05 means a difference this large would happen by chance less than 1 time in 20.

∇ *But we're about to test **every** word in the vocabulary: about 10,000 of them. If none of them really differed between the parties, roughly how many would still come out "significant" just by chance?*

To deal with this, we'll *adjust* the p-values for the number of tests (with a method called Benjamini–Hochberg). Only words that pass the adjusted test count as significant.

## Distinctive words: all at once

`p4p.distinctive_words` does everything above, for every word, in one step.

In [ ]:
#@title Find distinctive words
result = p4p.distinctive_words(counts_D, counts_R, names=('Democrat', 'Republican'))
result

| column | meaning |
|---|---|
| `count_…` | how many times each party used the word |
| `fpm_…` | frequency per million words, for each party |
| `obs_exp_Democrat` | Democrats' count divided by what we'd expect if both parties used the word equally (above 1 = more Democratic) |
| `log_ratio` | +1 = twice as frequent in Democratic speeches; −1 = twice as frequent in Republican ones |
| `g2`, `p` | the significance test, as above |
| `p_adj` | the p-value adjusted for testing thousands of words |
| `significant` | whether `p_adj` is below 0.05 |

Words used fewer than 5 times in total are left out. The most Democratic words are at the top of the table, the most Republican at the bottom.

In [ ]:
#@title Check: does the table agree with what we computed by hand?
result[result['word'] == 'immigration']

In [ ]:
#@title Only the significant words
significant = result[result['significant']]
significant

In [ ]:
#@title Plot the most distinctive words for each party
p4p.plot_distinctive_words(result)

∇ *What kinds of words are these? Can you sort them into types (people, places, topics, styles of speaking...)?*

∇ *Do any of them surprise you? Do any tell a story about the two parties?*

## Party, or period?

Look at "whilst" near the top of the Democratic list. Is "whilst" really a Democratic word?

The Republican party was only founded in the 1850s: the first Republican State of the Union was Lincoln's in 1861. But the Democrats' speeches go back to Jackson in 1829. So every word from the 1830s–1850s will look "Democratic".

In [ ]:
#@title When was "whilst" used?
p4p.plot_word_over_time(df, 'whilst', color='party')

In [ ]:
#@title Which speeches use "whilst" the most?
p4p.count_word_over_time(df, 'whilst').sort_values('count', ascending=False)

∇ *Which president uses "whilst" the most? What does this tell you about trusting a word's total count?*

To compare the parties fairly, we should only compare speeches from the period when both existed:

In [ ]:
#@title Compare the parties only since 1861
df_since_1861 = df[df['year'] >= 1861]

result_since_1861 = p4p.distinctive_words_by_group(df_since_1861, 'party', 'Democrat', 'Republican')
p4p.plot_distinctive_words(result_since_1861)

∇ *What changed? Are there still words that look like they belong to a period rather than a party?*

In [ ]:
#@title TODO: Compare the parties since a later year of your choice
df_since = df[df['year'] >= REPLACE_ME]

result_since = p4p.distinctive_words_by_group(df_since, 'party', 'Democrat', 'Republican')
p4p.plot_distinctive_words(result_since)

## Words over time

In [ ]:
#@title Plot relative usage of "immigration" over time
p4p.plot_word_over_time(df, 'immigration', color='party')

In [ ]:
#@title Compare two words over time
p4p.plot_word_over_time(df, ['war', 'peace'])

In [ ]:
#@title TODO: Plot a distinctive word from your results over time
p4p.plot_word_over_time(df, 'REPLACE_ME', color='party')

## Reading in context: Key Word in Context (KWIC)

Counting words tells us *where to look*. To understand what a word is doing, we have to read it.

In [ ]:
#@title Every use of "immigration", in context
p4p.kwic(df, 'immigration')

In [ ]:
#@title Only in one speech
p4p.kwic(df[df['id'] == 'Trump_2017'], 'immigration')

In [ ]:
#@title Phrases work too, and you can choose how much context to show
p4p.kwic(df, 'the american people', window=12)

In [ ]:
#@title TODO: Read a distinctive word from your results in context
p4p.kwic(df, 'REPLACE_ME')

∇ *Does reading the word in context confirm the story you told about it above, or complicate it?*

## Your turn: other comparisons

`p4p.distinctive_words_by_group` can compare any two groups in any column. We can also make new columns to compare.

In [ ]:
#@title Compare speeches before and after 1900
df['era'] = df['year'].apply(lambda year: 'before 1900' if year < 1900 else 'after 1900')

result_era = p4p.distinctive_words_by_group(df, 'era', 'before 1900', 'after 1900')
p4p.plot_distinctive_words(result_era)

In [ ]:
#@title TODO: Compare two presidents (check the `president` column for their exact names)
result_presidents = p4p.distinctive_words_by_group(df, 'president', 'REPLACE_ME', 'REPLACE_ME_ALSO')
p4p.plot_distinctive_words(result_presidents)

In [ ]:
#@title TODO: Use the interactive table (click the table icon next to the output) to sort and filter your results, then 'narrativize' or interpret some distinctive words
result_presidents

In [ ]:
#@title TODO: Plot one of those words over time, and read it in context